# Experiment 2: COCO val2017 Training (Optimized)

**Purpose:** Secondary benchmark (Table 2 in manuscript). 3 seeds minimum.

**Dataset:** COCO 2017 (118K train / 5K val, 80 classes)

**Expected time:** ~3-4 hours per seed on T4 with `--pretrained --compile` (~10-12 hours total)

> ⚠️ **Optimization:** We now use `--pretrained` and `--compile` to halve training time.

In [ ]:
# === Cell 1: Setup ===
!rm -rf /content/tinyYOLO 2>/dev/null
!git clone https://github.com/ShMazumder/tinyYOLO.git /content/tinyYOLO
%cd /content/tinyYOLO
!pip install -e . -q
!pip install tqdm timm -q

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# === Cell 2: Download Dataset ===
!python -c "from ultralytics.data.utils import check_det_dataset; check_det_dataset('coco128.yaml')"

DATA = 'coco128.yaml'  # Change to 'coco.yaml' for full COCO
print(f"\nUsing dataset: {DATA}")

In [ ]:
# === Cell 3: Configuration ===
SEEDS = [42, 123, 256]
IMGSZ = 416
EPOCHS = 150  # Reduced from 300
BATCH = 128   # Maximize T4
WARMUP = 3

print(f"Plan: {len(SEEDS)} seeds × 2 variants = {len(SEEDS)*2} runs")
print(f"Dataset: {DATA} | {IMGSZ}×{IMGSZ} | {EPOCHS} epochs | batch={BATCH}")

In [ ]:
# === Cell 4: Train QUANTIZED on COCO — 3 seeds ===
for i, seed in enumerate(SEEDS):
    name = f"coco-q-{IMGSZ}-seed{seed}"
    print(f"\n{'='*70}")
    print(f"  [{i+1}/{len(SEEDS)}] Quantized seed={seed}")
    print(f"{'='*70}", flush=True)
    get_ipython().system(
        f"python scripts/train.py --task det --variant quantized "
        f"--data {DATA} --imgsz {IMGSZ} --epochs {EPOCHS} --batch {BATCH} "
        f"--seed {seed} --warmup {WARMUP} --name {name} --pretrained --compile"
    )
print("\n🎉 Quantized COCO training complete!")

In [ ]:
# === Cell 5: Train STANDARD on COCO — 1 seed ===
print(f"{'='*70}")
print(f"  Standard seed=42")
print(f"{'='*70}", flush=True)
get_ipython().system(
    f"python scripts/train.py --task det --variant standard "
    f"--data {DATA} --imgsz {IMGSZ} --epochs {EPOCHS} --batch {BATCH} "
    f"--seed 42 --warmup {WARMUP} --name coco-std-{IMGSZ}-seed42 --pretrained --compile"
)
print("\n🎉 Standard COCO training complete!")